[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Tables and Queries &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the two tables the notebook's worked examples made:
five stations, one of them with no readings, and the year of readings, joined by `station_id`. Run it
first. The tasks do not depend on one another, and the last cell removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from contextlib import closing
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL,
        latitude REAL NOT NULL
    );

    CREATE TABLE readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL
    );
""")
station_ids = {}
for name, latitude in LATITUDES.items():
    station_ids[name] = build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((station_ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


**1.** North to south.


In [2]:
with closing(sqlite3.connect(DATABASE)) as conn:
    for name, latitude in conn.execute("SELECT name, latitude FROM stations ORDER BY latitude DESC"):
        print(f"{name:<9} {latitude:.2f}")


Svalbard  78.22
Kirkenes  69.73
Tromso    69.65
Bergen    60.39
Oslo      59.91


`DESC` puts the largest latitude first, and the largest latitude belongs to the station farthest
north.


**2.** Hours above 15 degrees, by name.


In [3]:
with closing(sqlite3.connect(DATABASE)) as conn:
    warm = conn.execute("""
        SELECT s.name, COUNT(*) AS hours
        FROM stations AS s
        JOIN readings AS r ON r.station_id = s.id
        WHERE r.celsius > 15
        GROUP BY s.id
        ORDER BY s.name
    """).fetchall()

print(warm)


[('Bergen', 1621), ('Oslo', 1043), ('Tromso', 140)]


A plain `JOIN` with the condition in `WHERE` suits this question, which asks only about stations that
went above 15 degrees. Svalbard never did, so none of its rows are left to make a group, and Kirkenes
has no readings to join.


**3.** Tromso's warmest readings, found by name.


In [4]:
with closing(sqlite3.connect(DATABASE)) as conn:
    warmest = conn.execute("""
        SELECT r.hour, r.celsius
        FROM readings AS r
        JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ? AND r.celsius IS NOT NULL
        ORDER BY r.celsius DESC, r.hour
        LIMIT 3
    """, ("Tromso",)).fetchall()

print(warmest)


[('2025-07-09T15:00', 16.2), ('2025-07-13T15:00', 16.2), ('2025-07-14T14:00', 16.2)]


The join lets `WHERE` test the name, which lives in `stations`, while the rows come from `readings`.
Tromso reached 16.2 degrees more than three times, so `r.hour` picks the earliest three of those
readings, and the answer is the same on every run.


**4.** Stations with no readings.


In [5]:
with closing(sqlite3.connect(DATABASE)) as conn:
    silent = conn.execute("""
        SELECT s.name
        FROM stations AS s
        LEFT JOIN readings AS r ON r.station_id = s.id
        WHERE r.id IS NULL
    """).fetchall()

print(silent)


[('Kirkenes',)]


The left join gives a station with no readings one row whose reading columns are all `NULL`, and
`WHERE r.id IS NULL` keeps exactly those rows. Here a `WHERE` on the joined table is what the
question needs, since the question is about the stations that found no match.


**5.** Svalbard's cold months, with `HAVING`.


In [6]:
with closing(sqlite3.connect(DATABASE)) as conn:
    months = conn.execute("""
        SELECT substr(r.hour, 1, 7) AS month, AVG(r.celsius) AS mean
        FROM readings AS r
        JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ?
        GROUP BY month
        HAVING AVG(r.celsius) < -8
        ORDER BY month
    """, ("Svalbard",)).fetchall()

for month, mean in months:
    print(month, f"{mean:.1f}")


2025-01 -13.4
2025-02 -12.4
2025-03 -9.3
2025-11 -8.7
2025-12 -12.1


`WHERE` keeps Svalbard's readings, `GROUP BY` puts them into months, and `HAVING` keeps the months
whose mean is below -8. Python's format rounds the mean to one decimal place as it prints it.


**6.** The statement SQLite kept.


In [7]:
with closing(sqlite3.connect(DATABASE)) as conn:
    (statement,) = conn.execute("SELECT sql FROM sqlite_schema WHERE name = ?", ("readings",)).fetchone()

print(statement)


CREATE TABLE readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL
    )


`sqlite_schema` keeps the `CREATE TABLE` statement much as it was written, line breaks and spacing
included, so the statement printed here keeps the layout the first cell gave it.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Tables and Queries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/03-tables-and-queries.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
